# SimCLR Ablation Studies

Two ablations on CIFAR-10 with a ResNet-18 encoder:

1. **Augmentation ablation** — linear eval accuracy with each augmentation removed in turn
2. **Temperature ablation** — linear eval accuracy vs NT-Xent temperature τ

> Enable GPU: *Notebook Settings → Accelerator → GPU T4 x1*

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
PRETRAIN_EPOCHS: int = 50
EVAL_EPOCHS: int = 40
BATCH_SIZE: int = 512
TEMPERATURE: float = 0.5  # default; overridden in temperature ablation
LR_PRETRAIN: float = 3e-4
LR_EVAL: float = 0.1
DATA_DIR: str = "/kaggle/working/data"
SAVE_DIR: Path = Path("/kaggle/working/plots")
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2023, 0.1994, 0.2010)

SAVE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "font.family": "sans-serif",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)

logger.info(f"Device: {DEVICE}")
logger.info(f"Pretrain epochs: {PRETRAIN_EPOCHS}  |  Eval epochs: {EVAL_EPOCHS}  |  Batch size: {BATCH_SIZE}")

## Data

Datasets are downloaded once and reused across all 11 runs.

In [ ]:
eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    ]
)

# Base datasets — PIL images, no transform; augmentation applied per-run in PairDataset.
cifar_train_raw = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=None)
cifar_test_eval = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=eval_transform)

# Eval-transform train set used for deterministic feature extraction.
cifar_train_eval = datasets.CIFAR10(DATA_DIR, train=True, download=False, transform=eval_transform)

logger.info(f"Train: {len(cifar_train_raw):,}  |  Test: {len(cifar_test_eval):,}")

## Model

In [ ]:
class PairDataset(torch.utils.data.Dataset):
    """Wraps a PIL dataset and applies augmentation independently twice per sample.

    Args:
        dataset: Base dataset returning (PIL image, label).
        augmentation: Transform applied independently to produce each view.
    """

    def __init__(self, dataset: torch.utils.data.Dataset, augmentation: transforms.Compose) -> None:
        self.dataset = dataset
        self.augmentation = augmentation

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, int]:
        image, label = self.dataset[idx]
        return self.augmentation(image), self.augmentation(image), label


class ProjectionHead(nn.Module):
    """3-layer MLP: Linear → BN → ReLU → Linear → BN → ReLU → Linear.

    Args:
        in_dim: Input dimension.
        hidden_dim: Hidden layer width.
        out_dim: Output embedding dimension.
    """

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SimCLR(nn.Module):
    """SimCLR: ResNet-18 encoder with a non-linear projection head.

    Args:
        out_dim: Projection head output dimension.
    """

    def __init__(self, out_dim: int = 128) -> None:
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.encoder_dim: int = backbone.fc.in_features  # 512
        backbone.fc = nn.Identity()
        self.encoder = backbone
        self.projector = ProjectionHead(self.encoder_dim, self.encoder_dim, out_dim)

    def forward(self, x: torch.Tensor, project: bool = True) -> torch.Tensor:
        """Run encoder, optionally through the projection head.

        Args:
            x: Input tensor of shape (B, 3, H, W).
            project: Return projector output if True, encoder output if False.

        Returns:
            Feature tensor of shape (B, out_dim) or (B, encoder_dim).
        """
        h = self.encoder(x)
        return self.projector(h) if project else h


def nt_xent_loss(z_i: torch.Tensor, z_j: torch.Tensor, temperature: float) -> torch.Tensor:
    """Normalized temperature-scaled cross-entropy loss (NT-Xent).

    Treats each sample's partner augmentation as its sole positive and all
    other 2(N-1) views in the batch as negatives.

    Args:
        z_i: Projection embeddings for view i, shape (N, D).
        z_j: Projection embeddings for view j, shape (N, D).
        temperature: Scaling factor τ.

    Returns:
        Scalar mean loss.
    """
    n = z_i.shape[0]
    z = F.normalize(torch.cat([z_i, z_j], dim=0), dim=1)  # (2N, D)
    sim = (z @ z.T) / temperature  # (2N, 2N)

    # Zero out self-similarity on the diagonal
    mask = torch.eye(2 * n, dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, float("-inf"))

    # Positive for sample i is i+N; for i+N it is i
    targets = torch.cat(
        [
            torch.arange(n, 2 * n, device=z.device),
            torch.arange(0, n, device=z.device),
        ]
    )
    return F.cross_entropy(sim, targets)

## Training and Evaluation Helpers

In [ ]:
def build_augmentation(
    use_crop: bool = True,
    use_jitter: bool = True,
    use_grayscale: bool = True,
    use_blur: bool = True,
    use_flip: bool = True,
) -> transforms.Compose:
    """Build a SimCLR augmentation pipeline with individually togglable components.

    Args:
        use_crop: Include RandomResizedCrop.
        use_jitter: Include ColorJitter.
        use_grayscale: Include RandomGrayscale.
        use_blur: Include GaussianBlur.
        use_flip: Include RandomHorizontalFlip.

    Returns:
        Composed transform mapping a PIL image to a normalised tensor.
    """
    t: list[transforms.Transform] = []

    if use_crop:
        t.append(
            transforms.RandomResizedCrop(
                32,
                scale=(0.2, 1.0),
                interpolation=InterpolationMode.BICUBIC,
            )
        )
    else:
        t.append(transforms.Resize(32))  # keep spatial size consistent

    if use_flip:
        t.append(transforms.RandomHorizontalFlip(p=0.5))

    if use_jitter:
        t.append(
            transforms.RandomApply(
                [transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)], p=0.8
            )
        )

    if use_grayscale:
        t.append(transforms.RandomGrayscale(p=0.2))

    if use_blur:
        # kernel_size must be odd; 3 is appropriate for 32x32 CIFAR images
        t.append(transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5))

    t += [
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    ]
    return transforms.Compose(t)


def pretrain(
    augmentation: transforms.Compose,
    temperature: float = TEMPERATURE,
) -> SimCLR:
    """Pretrain a SimCLR model with the given augmentation and temperature.

    Uses mixed precision (autocast + GradScaler) when a CUDA device is available.

    Args:
        augmentation: Augmentation pipeline used to generate positive pairs.
        temperature: NT-Xent temperature τ.

    Returns:
        Pretrained SimCLR model moved to CPU.
    """
    loader = DataLoader(
        PairDataset(cifar_train_raw, augmentation),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        drop_last=True,
    )

    model = SimCLR(out_dim=128).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_PRETRAIN, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=PRETRAIN_EPOCHS,
        eta_min=1e-6,
    )
    scaler = GradScaler(enabled=DEVICE.type == "cuda")

    model.train()
    for epoch in range(1, PRETRAIN_EPOCHS + 1):
        total = 0.0
        for v_i, v_j, _ in loader:
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=DEVICE.type == "cuda"):
                z_i = model(v_i.to(DEVICE), project=True)
                z_j = model(v_j.to(DEVICE), project=True)
                loss = nt_xent_loss(z_i, z_j, temperature)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total += loss.item()
        scheduler.step()
        if epoch % 10 == 0 or epoch == PRETRAIN_EPOCHS:
            logger.info(f"  Pretrain [{epoch:3d}/{PRETRAIN_EPOCHS}] " f"loss={total / len(loader):.4f}")

    return model.cpu()


@torch.no_grad()
def extract_features(
    encoder: nn.Module,
    loader: DataLoader,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Extract encoder features for an entire dataset.

    Sets the encoder to eval mode for the duration of extraction to ensure
    BatchNorm uses running statistics, then restores the original mode.

    Args:
        encoder: Encoder on DEVICE.
        loader: DataLoader yielding (image, label) batches.

    Returns:
        Tuple of (features, labels) stored on CPU.
    """
    was_training = encoder.training
    encoder.eval()
    feats, labs = [], []
    for images, labels in loader:
        feats.append(encoder(images.to(DEVICE)).flatten(1).cpu())
        labs.append(labels)
    if was_training:
        encoder.train()
    return torch.cat(feats), torch.cat(labs)


def linear_eval(model: SimCLR) -> float:
    """Freeze the encoder and train a linear classifier on CIFAR-10.

    Features are extracted once with deterministic transforms and cached in
    memory. Does not modify the state of the input model.

    Args:
        model: Pretrained SimCLR model (CPU).

    Returns:
        Top-1 test accuracy as a float in [0, 1].
    """
    train_loader = DataLoader(
        cifar_train_eval,
        batch_size=512,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
    )
    test_loader = DataLoader(
        cifar_test_eval,
        batch_size=512,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
    )

    encoder = model.encoder.to(DEVICE)
    for p in encoder.parameters():
        p.requires_grad = False

    logger.info("  Extracting features...")
    train_feats, train_labs = extract_features(encoder, train_loader)
    test_feats, test_labs = extract_features(encoder, test_loader)
    encoder.cpu()

    classifier = nn.Linear(model.encoder_dim, 10).to(DEVICE)
    optimizer = torch.optim.SGD(
        classifier.parameters(),
        lr=LR_EVAL,
        momentum=0.9,
        nesterov=True,
        weight_decay=0.0,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EVAL_EPOCHS,
        eta_min=1e-4,
    )
    loss_fn = nn.CrossEntropyLoss()
    feat_loader = DataLoader(
        TensorDataset(train_feats, train_labs),
        batch_size=512,
        shuffle=True,
    )

    classifier.train()
    for _ in range(EVAL_EPOCHS):
        for feats, labels in feat_loader:
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(classifier(feats.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
        scheduler.step()

    classifier.eval()
    with torch.no_grad():
        preds = classifier(test_feats.to(DEVICE)).argmax(1).cpu()
    return (preds == test_labs).float().mean().item()

## Augmentation Ablation

SimCLR is retrained from scratch with each augmentation removed in turn, then evaluated with the linear protocol. Results are saved immediately after this cell completes.

In [ ]:
conditions: dict[str, dict[str, bool]] = {
    "All augs (baseline)": {},
    "Remove crop": {"use_crop": False},
    "Remove color jitter": {"use_jitter": False},
    "Remove grayscale": {"use_grayscale": False},
    "Remove Gaussian blur": {"use_blur": False},
    "Remove flip": {"use_flip": False},
}

aug_results: dict[str, float] = {}
for name, kwargs in conditions.items():
    logger.info(f"=== Augmentation ablation: {name} ===")
    model = pretrain(build_augmentation(**kwargs))
    acc = linear_eval(model) * 100
    aug_results[name] = acc
    logger.info(f"  → {name}: {acc:.2f}%")
    del model
    torch.cuda.empty_cache()

# --- Plot ---
baseline = aug_results["All augs (baseline)"]
names = list(aug_results.keys())
accs = list(aug_results.values())
colors = ["steelblue" if n == "All augs (baseline)" else "tomato" for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, accs, color=colors, edgecolor="white", height=0.6)

for bar, acc in zip(bars, accs):
    drop = baseline - acc
    label = f"{acc:.1f}%"
    if drop > 0.05:
        label += f"  (\u2212{drop:.1f}pp)"
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, label, va="center", fontsize=9)

ax.axvline(baseline, linestyle="--", color="steelblue", alpha=0.6, label=f"Baseline {baseline:.1f}%")
ax.set_xlabel("Top-1 Test Accuracy (%)")
ax.set_title(
    f"SimCLR Augmentation Ablation \u2014 CIFAR-10\n" f"({PRETRAIN_EPOCHS} pretrain epochs, {EVAL_EPOCHS} eval epochs)"
)
ax.set_xlim(0, max(accs) + 8)
ax.legend(fontsize=9)
ax.invert_yaxis()
fig.tight_layout()

path = SAVE_DIR / "augmentation_ablation.png"
fig.savefig(path, dpi=150)
plt.show()
plt.close(fig)
logger.info(f"Saved: {path}")

## Temperature Ablation

The full augmentation pipeline is fixed. SimCLR is retrained at each temperature τ and evaluated with the linear protocol.

In [ ]:
temperatures = [0.1, 0.3, 0.5, 0.7, 1.0]
full_aug = build_augmentation()

temp_results: dict[float, float] = {}
for tau in temperatures:
    logger.info(f"=== Temperature ablation: \u03c4={tau} ===")
    model = pretrain(full_aug, temperature=tau)
    acc = linear_eval(model) * 100
    temp_results[tau] = acc
    logger.info(f"  \u2192 \u03c4={tau}: {acc:.2f}%")
    del model
    torch.cuda.empty_cache()

# --- Plot ---
taus = list(temp_results.keys())
accs = list(temp_results.values())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(taus, accs, marker="o", color="steelblue", markersize=8)

for tau, acc in zip(taus, accs):
    ax.annotate(f"{acc:.1f}%", (tau, acc), textcoords="offset points", xytext=(5, 6), fontsize=9)

ax.axvline(TEMPERATURE, linestyle="--", color="gray", alpha=0.6, label=f"Paper default \u03c4={TEMPERATURE}")
ax.set_xlabel("Temperature \u03c4")
ax.set_ylabel("Top-1 Test Accuracy (%)")
ax.set_title(
    f"SimCLR Temperature Sensitivity \u2014 CIFAR-10\n"
    f"({PRETRAIN_EPOCHS} pretrain epochs, {EVAL_EPOCHS} eval epochs)"
)
ax.set_xticks(taus)
ax.set_ylim(0, max(accs) + 10)
ax.legend(fontsize=9)
fig.tight_layout()

path = SAVE_DIR / "temperature_ablation.png"
fig.savefig(path, dpi=150)
plt.show()
plt.close(fig)
logger.info(f"Saved: {path}")